In [ ]:
%matplotlib inline

from pathlib import Path
import sys


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "scripts" / "download_artifact.py").is_file():
            return cand
    raise FileNotFoundError("Could not find repo root. Run this notebook from inside the repository.")


def _script_dir() -> Path:
    root = _repo_root()
    for d in (Path.cwd().resolve(), root / "analysis" / "pythia"):
        if (d / "pareto_radar_comparison.py").is_file():
            return d
    raise FileNotFoundError(
        "Place pareto_radar_comparison.py in analysis/pythia, or run the kernel from inside the repository."
    )


REPO_ROOT = _repo_root()
SCRIPT_DIR = _script_dir()
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import pareto_radar_comparison as prc

In [ ]:
METRICS = prc.METRICS
EXPERIMENTS = prc.EXPERIMENTS
DATA_RESULTS_DIR = prc.DATA_RESULTS_DIR
DATA_MODEL_WEIGHTS_DIR = prc.DATA_MODEL_WEIGHTS_DIR

all_data = {}
all_metrics_values = {m: [] for m in METRICS}

for exp_name, exp_info in EXPERIMENTS.items():
    print(f"Loading {exp_name}...")
    eval_dir = DATA_RESULTS_DIR / exp_info["eval_dir"]
    models_dir = DATA_MODEL_WEIGHTS_DIR / exp_info["models_dir"]

    if not eval_dir.exists() or not models_dir.exists():
        print("  Skipping — directories not found")
        continue

    df = prc.load_experiment_data(eval_dir, models_dir)
    df = df.dropna(subset=METRICS)

    if df.empty:
        print("  Skipping — no complete data")
        continue

    df = prc.compute_pareto_front(df, METRICS)
    all_data[exp_name] = df

    for m in METRICS:
        all_metrics_values[m].extend(df[m].tolist())

    n_pareto = int(df["pareto_optimal"].sum())
    print(f"  Loaded {len(df)} trainers, {n_pareto} Pareto optimal")

if not all_data:
    raise RuntimeError("No experiment data loaded. Check data_results/ and data_model_weights/ under the repo root.")

global_ranges = {}
for m in METRICS:
    if all_metrics_values[m]:
        global_ranges[m] = (min(all_metrics_values[m]), max(all_metrics_values[m]))

print("\nGlobal metric ranges:")
for m, (lo, hi) in global_ranges.items():
    print(f"  {m}: [{lo:.3f}, {hi:.3f}]")

In [ ]:
output_path = SCRIPT_DIR / "my_plots" / "pareto_comparison"
output_path.mkdir(parents=True, exist_ok=True)

prc.plot_combined_radar_labeled(all_data, output_path, global_ranges, save=False)

In [ ]:
prc.plot_best_comparison_labeled(all_data, output_path, global_ranges, top_n=3, save=False)